# Tahap 1 - Pengumpulan Data Ulasan Google Play Store

Notebook ini digunakan untuk mengumpulkan ulasan publik aplikasi JakOne Mobile dari Google Play Store sebagai data awal penelitian **Analisis Sentimen pada Aplikasi JakOne Mobile Menggunakan Metode IndoBERT**.

Tahap ini hanya mengambil data mentah. Tidak ada preprocessing teks, pelabelan sentimen, split dataset, atau training model.

## Konfigurasi

Target pengambilan data adalah 140000 ulasan untuk rentang tahun 2022 sampai 2026. Jika ulasan tersedia kurang dari target, proses tetap menyimpan data yang berhasil dikumpulkan.

In [ ]:
from pathlib import Path
from time import sleep

import pandas as pd
from google_play_scraper import Sort, app, reviews


APP_ID = "com.dev.jakone.mbanking"
TARGET_REVIEWS = 140000
START_YEAR = 2022
END_YEAR = 2026
OUTPUT_PATH = "data/raw/jakone_reviews_raw.csv"

LANGUAGE = "id"
COUNTRY = "id"
BATCH_SIZE = 200
REQUEST_DELAY_SECONDS = 1
MAX_RETRIES = 3

OUTPUT_COLUMNS = [
    "review_id",
    "review",
    "rating",
    "review_date",
    "app_version",
    "thumbs_up_count",
    "year",
]

: 

## Validasi Metadata dan Fungsi Pengambilan Data

Metadata aplikasi dicek lebih dulu menggunakan `app()`. Jika package name tidak valid atau metadata gagal diambil, proses dihentikan sebelum mengambil ulasan.

In [ ]:
def validate_app_metadata():
    try:
        metadata = app(APP_ID, lang=LANGUAGE, country=COUNTRY)
    except Exception as exc:
        print("\nStatus metadata aplikasi: GAGAL")
        print(f"Package name yang digunakan: {APP_ID}")
        print(f"Error saat mengambil metadata aplikasi: {exc}")
        print("Proses dihentikan karena metadata aplikasi tidak dapat divalidasi.")
        return None

    print("\nStatus metadata aplikasi: BERHASIL")
    print(f"Nama aplikasi: {metadata.get('title', '-')}")
    print(f"Developer: {metadata.get('developer', '-')}")
    print(f"Rating aplikasi: {metadata.get('score', '-')}")
    print(f"Jumlah review: {metadata.get('reviews', '-')}")
    print(f"Package name yang digunakan: {metadata.get('appId', APP_ID)}")
    return metadata


def fetch_reviews():
    all_reviews = []
    continuation_token = None
    batch_number = 1

    while len(all_reviews) < TARGET_REVIEWS:
        remaining = TARGET_REVIEWS - len(all_reviews)
        batch_count = min(BATCH_SIZE, remaining)

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                batch, continuation_token = reviews(
                    APP_ID,
                    lang=LANGUAGE,
                    country=COUNTRY,
                    sort=Sort.NEWEST,
                    count=batch_count,
                    continuation_token=continuation_token,
                )
                break
            except Exception as exc:
                print(
                    f"Gagal mengambil batch {batch_number} "
                    f"(percobaan {attempt}/{MAX_RETRIES}): {exc}"
                )
                if attempt == MAX_RETRIES:
                    print("Pengambilan data dihentikan. Menyimpan data yang sudah terkumpul.")
                    return all_reviews
                sleep(REQUEST_DELAY_SECONDS * attempt)

        if not batch:
            if batch_number == 1:
                print("Batch pertama kosong. Kemungkinan penyebab:")
                print("- package name salah")
                print("- aplikasi tidak tersedia di region tersebut")
                print("- Google Play membatasi akses sementara")
                print("- tidak ada review yang bisa diambil oleh library")
            else:
                print("Tidak ada ulasan tambahan dari Google Play Store.")
            break

        all_reviews.extend(batch)
        print(f"Batch {batch_number}: total ulasan terkumpul = {len(all_reviews)}")
        batch_number += 1

        if continuation_token is None:
            print("Continuation token habis. Semua ulasan yang tersedia sudah diambil.")
            break

        sleep(REQUEST_DELAY_SECONDS)

    return all_reviews

## Pengambilan Data Mentah

In [ ]:
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

metadata = validate_app_metadata()
if metadata is None:
    raise SystemExit("Metadata aplikasi gagal divalidasi. Pengambilan review dihentikan.")


raw_reviews = fetch_reviews()
df_raw = pd.DataFrame(raw_reviews)
total_before_year_filter = len(df_raw)

print(f"Jumlah data sebelum filter tahun: {total_before_year_filter}")

## Penyesuaian Kolom dan Filter Tahun

Kolom dari `google-play-scraper` disesuaikan dengan kebutuhan dataset penelitian.

In [ ]:
if df_raw.empty:
    df_reviews = pd.DataFrame(columns=OUTPUT_COLUMNS)
    total_after_year_filter = 0
    total_after_deduplication = 0
else:
    rename_columns = {
        "reviewId": "review_id",
        "content": "review",
        "score": "rating",
        "at": "review_date",
        "reviewCreatedVersion": "app_version",
        "thumbsUpCount": "thumbs_up_count",
    }
    df_reviews = df_raw.rename(columns=rename_columns)

    for column in OUTPUT_COLUMNS:
        if column not in df_reviews.columns and column != "year":
            df_reviews[column] = pd.NA

    df_reviews["review_date"] = pd.to_datetime(df_reviews["review_date"], errors="coerce")
    df_reviews["year"] = df_reviews["review_date"].dt.year
    df_reviews["app_version"] = df_reviews["app_version"].fillna("")
    df_reviews["review"] = df_reviews["review"].fillna("")

    df_reviews = df_reviews[OUTPUT_COLUMNS]
    df_reviews = df_reviews[df_reviews["year"].between(START_YEAR, END_YEAR, inclusive="both")]
    total_after_year_filter = len(df_reviews)

    df_reviews = df_reviews.drop_duplicates(subset="review_id", keep="first")
    total_after_deduplication = len(df_reviews)
    df_reviews = df_reviews.sort_values("review_date", ascending=False).reset_index(drop=True)

print(f"Jumlah data setelah filter tahun: {total_after_year_filter}")
print(f"Jumlah data setelah hapus duplikasi: {total_after_deduplication}")

## Simpan Dataset Mentah

In [ ]:
df_reviews.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Data disimpan ke: {OUTPUT_PATH}")

## Ringkasan Data

In [ ]:
print("Distribusi data per tahun:")
if df_reviews.empty:
    print("Tidak ada data pada rentang tahun yang ditentukan.")
else:
    print(df_reviews["year"].value_counts().sort_index())

print("\n5 baris pertama:")
display(df_reviews.head())